In [40]:
import torch
import numpy as np


MSKint32t = int
PRINTLEVEL = 0
LMD_TOL = 1e-4

def set_print_level(level):
    global PRINTLEVEL
    PRINTLEVEL = level

def construct_P_matrix(minimize_order, segment_num, poly_order, room_time, MQM, ptype):
    min_order_l = int(np.floor(minimize_order))
    min_order_u = int(np.ceil(minimize_order))

    NUMQNZ = 0
    NUMQ_blk = poly_order + 1  # default minimize the jerk and minimize_order = 3
    if ptype.lower() == "f":
        NUMQNZ = segment_num * 3 * (NUMQ_blk ** 2)
    else:
        NUMQNZ = segment_num * 3 * NUMQ_blk * (NUMQ_blk + 1) / 2

    qval = torch.zeros(int(NUMQNZ),dtype=torch.double)
    qsubi = np.zeros(int(NUMQNZ),dtype=int)
    qsubj = np.zeros(int(NUMQNZ),dtype=int)

    sub_shift = 0
    idx = 0
    s1d1CtrlP_num = poly_order + 1
    s1CtrlP_num = 3 * s1d1CtrlP_num

    for k in range(segment_num):
        scale_k = room_time[k]
        for p in range(3):
            for i in range(s1d1CtrlP_num):
                for j in range(s1d1CtrlP_num):
                    if ((ptype.lower() == "l" and i >= j) or (ptype.lower() == "u" and i <= j) or ptype.lower() == "f"):
                        qsubi[idx] = sub_shift + p * s1d1CtrlP_num + i
                        qsubj[idx] = sub_shift + p * s1d1CtrlP_num + j

                        if min_order_l == min_order_u:
                            qval[idx] = MQM[i][j] / (scale_k ** (2 * min_order_u - 3))
                        else:
                            qval[idx] = ((minimize_order - min_order_l) / (scale_k ** (2 * min_order_u - 3)) +
                                         (min_order_u - minimize_order) / (scale_k ** (2 * min_order_l - 3))) * MQM[i][j]
                        idx += 1

        sub_shift += s1CtrlP_num

    return qval, qsubi, qsubj



In [52]:
minimize_order=3.0
segment_num=3
poly_order=6
room_time= torch.tensor([3.80504758, 1.14997985, 0.83818007],dtype=torch.double)

MQM=[[  2057.14285714,  -5142.85714286,   3497.14285714,   -102.85714286,
    -102.85714286,   -102.85714286,   -102.85714286],
 [ -5142.85714286,  13577.14285714, -10182.85714286,    617.14285714,
     617.14285714,    617.14285714,   -102.85714286],
 [  3497.14285714, -10182.85714286,  9257.14285714,  -1542.85714286,
   -1542.85714286,    617.14285714,   -102.85714286],
 [  -102.85714286,    617.14285714,  -1542.85714286,   2057.14285714,
   -1542.85714286,    617.14285714,   -102.85714286],
 [  -102.85714286,   617.14285714,  -1542.85714286,  -1542.85714286,
    9257.14285714, -10182.85714286,   3497.14285714],
 [  -102.85714286,    617.14285714,    617.14285714,    617.14285714,
  -10182.85714286,  13577.14285714,  -5142.85714286],
 [  -102.85714286,   -102.85714286,   -102.85714286,   -102.85714286,
    3497.14285714,  -5142.85714286,   2057.14285714]]
ptype = "l"

myval, myrow, mycol= construct_P_matrix(minimize_order, segment_num, poly_order, room_time, MQM, ptype)
from scipy.sparse import coo_matrix
p_coomatrix = torch.sparse_coo_tensor((myrow, mycol), myval)
len(myrow)

252

In [173]:
import torch
import numpy as np
from scipy.sparse import coo_matrix
import time
T=10000
s=time.time()
row = np.arange(T)
col = np.arange(T)
c=np.ones(T)
val=np.ones(T)
p = coo_matrix((val,(row, col)))

p=p.dot(c)
# # print("time", time.time()-s)
# p.dot(c)

print(time.time()-s)


0.0007946491241455078


In [172]:
v=time.time()
D = np.eye(T)
D @ c
print("time", time.time()-v)

time 0.037537336349487305


In [195]:
import torch
import numpy as np
from scipy.sparse import coo_matrix
v=time.time()
T=10000

row = np.arange(T)
col = np.arange(T)
c=torch.ones(T)
val=torch.ones(T,dtype=torch.double)
p = torch.sparse_coo_tensor((row, col),val)

p1=p.multiply(c)
print("time", time.time()-v)

time 0.0027399063110351562


In [196]:
v=time.time()
D = torch.eye(T)
D@c
print("time", time.time()-v)

time 0.07256722450256348


In [197]:
p1.coalesce().values()

tensor([1., 1., 1.,  ..., 1., 1., 1.], dtype=torch.float64)

In [284]:
row=[0,0,1]
col=[0,1,0]
val=torch.ones(3,dtype=torch.double)
p=torch.sparse_coo_tensor((row, col),val)

c=torch.tensor([2*[1]],dtype=torch.double)
p_csr = p.to_sparse_csr()
# result = torch.sparse.mm(p_csr, c.T)
result = torch.mm(p, c.T)
print(result)


tensor([[2.],
        [1.]], dtype=torch.float64)


In [241]:
torch.tensor([10*[0]],dtype=torch.double)

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=torch.float64)

In [6]:
import numpy as np

print(type(np.array([1])))

<class 'numpy.ndarray'>
